In [83]:
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_nuclei_labels_output_dir,
    load_precomputed_nuclei_labels_if_available,
)
from utils.segmentation import predict_nuclei_labels

from scipy.ndimage import distance_transform_edt
from scipy.ndimage import binary_closing, binary_fill_holes
from skimage.morphology import ball

import tifffile
import napari

In [84]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor) 
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor) – FRET
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor) - used for nuclei segmentation
MARKERS = (("edCerulean_CTRL", 0), ("edCitrine_FRET", 1), ("edCitrine_CTRL", 2), ("brightfield", 3))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 0 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 0

In [85]:
# Iterate through the .lif files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

lif_containers

['..\\raw_data\\20260129_ABACUS timepoints_mock 3h.lif',
 '..\\raw_data\\20260129_ABACUS timepoints_sorb 3h.lif',
 '..\\raw_data\\20260204_Splitplate_mock_6h.lif',
 '..\\raw_data\\20260204_Splitplate_sorb_6h.lif']

In [86]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

Image name: Col0 1, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (115, 4, 256, 1024)
Image name: Col0 2, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (116, 4, 256, 1024)
Image name: Col0 3, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (135, 4, 256, 1024)
Image name: Col0 4, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (125, 4, 256, 1024)
Image name: Col0 5, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (107, 4, 256, 1024)
Image name: Col0 6, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (114, 4, 256, 1024)
Image name: Col0 7, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (118, 4, 256, 1024)


In [87]:
# Initialize Napari Viewer
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

[<Image layer 'edCerulean_CTRL' at 0x18263f82bf0>,
 <Image layer 'edCitrine_FRET' at 0x1826bef5e70>,
 <Image layer 'edCitrine_CTRL' at 0x1826bf22fb0>,
 <Image layer 'brightfield' at 0x1826bf5fc70>]

In [88]:
from skimage.filters import difference_of_gaussians

In [89]:
# Create new input array for ConvPaint segmentation ("negative" of brightfield + edCitrine_CTRL) with 1 to 99 percentile normalization

import numpy as np

def normalize_percentile(image, pmin=1, pmax=99):
    vmin, vmax = np.percentile(image, (pmin, pmax))
    image = np.clip(image, vmin, vmax)
    if vmax > vmin:
        image = (image - vmin) / (vmax - vmin)
    else:
        image = image * 0  # avoid division by zero; returns zeros
    return image

brightfield_norm = normalize_percentile(lif_image[3])
edcitrine_ctrl_norm = normalize_percentile(lif_image[2])

bf_dog = difference_of_gaussians(brightfield_norm, low_sigma=1, high_sigma=2) # Remove brightfield haze
bf_inv = 1 - bf_dog

convpaint_input = ((1 - brightfield_norm) + edcitrine_ctrl_norm) / 2

# Add convpaint input to napari viewer
viewer.add_image(convpaint_input, 
                name="convpaint_input",
                colormap="gray",
                blending="additive")
           

<Image layer 'convpaint_input' at 0x18239df1000>

In [90]:
import pyclesperanto_prototype as cle

root_labels = cle.voronoi_otsu_labeling(convpaint_input, spot_sigma=100, outline_sigma=20)

viewer.add_labels(root_labels, name="voronoi_otsu_labeling")

<Labels layer 'voronoi_otsu_labeling' at 0x1826c66fee0>

In [91]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_nuclei_labels_output_dir(RAW_DATA_DIRECTORY, lif_container_id)
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_nuclei_labels_if_available(nuclei_labels_dir, lif_image_name)

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)

# Add the prediction to the napari viewer
viewer.add_labels(nuclei_labels)

Nuclei labels directory: ..\raw_data\nuclei_labels\20260129_ABACUS timepoints_mock 3h
x_um: 0.7568359375, y_um: 0.75461640625, z_um: 0.399525
Rescale factor: 0.5286637076611433
Predictions already calculated for: Col0 1 ...loading


<Labels layer 'nuclei_labels' at 0x1826c145cc0>

In [92]:
nuclei_mask = nuclei_labels > 0
dist = distance_transform_edt(~nuclei_mask)

In [93]:
dist_mask = dist < 15
viewer.add_image(dist_mask, name="distance_transform", colormap="gray", blending="additive", opacity=0.5)

<Image layer 'distance_transform' at 0x1826c92da50>

In [94]:
root_labels = cle.pull(root_labels)

In [95]:
dist_mask.dtype

dtype('bool')

In [96]:
root_labels.dtype

dtype('uint32')

In [97]:
root_mask = dist_mask | (root_labels > 0)

In [98]:
viewer.add_image(root_mask, name="root_mask", colormap="gray", blending="additive", opacity=0.5)

<Image layer 'root_mask' at 0x18280ea7ca0>

In [ ]:
from scipy.ndimage import binary_fill_holes

# Apply binary closing

# Fill holes in the mask
root_mask = binary_fill_holes(root_mask)
viewer.add_image(root_mask, name="root_mask", colormap="gray", blending="additive", opacity=0.5)

<Image layer 'root_mask [3]' at 0x18281b12b60>

In [104]:
from skimage.morphology import binary_erosion, ball

# Perform binary erosion of root_mask by 2 pixels
eroded_root_mask = binary_erosion(root_mask, footprint=ball(2))
viewer.add_image(eroded_root_mask, name="root_mask_eroded", colormap="gray", blending="additive", opacity=0.5)

<Image layer 'root_mask_eroded' at 0x182ce82ead0>

In [110]:
from scipy.ndimage import distance_transform_edt
import numpy as np

# Compute the Euclidean distance from each nuclei_label pixel to the nearest background (0-valued) pixel in eroded_root_mask
distance_to_bg = distance_transform_edt(nuclei_labels == eroded_root_mask[0])

# Prepare an empty array to hold the per-nucleus distance values
nuclei_label_distances = np.zeros_like(nuclei_labels, dtype=distance_to_bg.dtype)

# Assign the minimum distance value inside each nucleus label to all its pixels
for label_id in np.unique(nuclei_labels):
    if label_id == 0:
        continue  # Skip background
    mask = nuclei_labels == label_id
    min_distance = distance_to_bg[mask].min() if np.any(mask) else 0
    nuclei_label_distances[mask] = min_distance

viewer.add_image(nuclei_label_distances, name="nuclei_dist_to_bg", colormap="viridis", blending="additive", opacity=0.6)

<Image layer 'nuclei_dist_to_bg [1]' at 0x18280f17d00>

In [109]:
viewer.add_image(nuclei_label_distances, name="nuclei_dist_to_bg", colormap="viridis", blending="additive", opacity=0.6)

<Image layer 'nuclei_dist_to_bg' at 0x18263e5a9e0>